### tutorial on running inference on custom inputs

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
os.chdir('..')

In [3]:
os.chdir('pi-rldif-ft')

In [4]:
# Note here are the pdbs from the sabdab test set used in AntiDIF / AntiFold/ ABMPNN
test_ids = pd.read_csv('data/raw_data/test_pdb_ids', index_col=0)
len(test_ids)

388

### First we create a custom input csv file pointing us to the protein we want to run inference on


Once you have the  the pdb ids of the prots of intrest, create the custom csv file used for inpu. 
We need the directory that are pdb files will be in (pdb_file_dir)


In [5]:

def find_inps_csv(pdb_ids, dir, all_chains=True):
    resut = {}
    resut['pdb_paths'] = [dir + iden + '.pdb' for iden in pdb_ids]
    if all_chains:
        resut['chains'] = ['all'] * len(pdb_ids)
    else: 
        raise NotImplementedError('only all chains support so far')
    return pd.DataFrame(resut)

pdb_file_dir = '/Users/nik/Documents/tf_ox/code/inv_folding/data/sabdab/' #dir pdbs will be in

sab_test_inps = find_inps_csv(test_ids['0'], pdb_file_dir)
#sab_test_inps.to_csv("data/raw_data/my_custom_input_csv.csv")


Then simply save the df and update the path in config.yaml for custom_pdb_input to your custom csv.

In the rest of this tutorial notebook we run inference using /pi-rldif-ft/data/raw_data/example.csv **to run with your data make sure to update  custom_pdb_input in config.yaml**

# Model inference 


In [6]:
from model.mod_pifold import InverseFoldingDiffusionPiFoldModel
from data.dataset import RLDIFDataset
from utils.utils import load_config
from torch.utils.data import DataLoader
import pickle as pkl
import torch

In [7]:
args = load_config('./configs/config.yaml')
master_config = load_config('./configs/master_config.yaml')

args.data.docker = False
args.data.rldif = args.rldif
args.data.protein_mpnn = args.protein_mpnn

args.data.custom_pdb_input

'data/raw_data/example.csv'

In [8]:
#test (have to change config.yaml for this)
custom_ds = RLDIFDataset(args.data) 

100%|██████████| 1/1 [00:00<00:00, 149.81it/s]

Entered 1 samples into Redis


In [9]:
#load model and weights 

## load model for inf
device = 'cpu'

args.pifold_model.free_positions = args.free_positions
model = InverseFoldingDiffusionPiFoldModel(args.pifold_model).to(device)
m_path = args.m_name
state_dict = torch.load(m_path, map_location=device)['state_dict'] #n.b added map_location 
new_state_dict = {}
for k, v in state_dict.items():
    new_state_dict[k.replace('model.', '')] = v
model.load_state_dict(new_state_dict)

collate_function = model.collate_fn
dataloader = DataLoader(
                custom_ds,
                batch_size=1, 
                shuffle=False,
                collate_fn=collate_function,
            )

In [10]:
# main inference run
for batch in dataloader:
    out = model.sample(batch.clone().to(device), closure=True)

/Users/nik/Documents/code/ox_re/AntiDIF/pi-rldif-ft/model/mod_pifold.py:198: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch.batch = torch.tensor(torch.cat(indices))


In [11]:
alphabet = "ACDEFGHIKLMNPQRSTVWY"
seqs = {}
for name, fp, ft, mask in zip(batch["names"], out["features_0_step"],
                              out["features_true"], out["mask"]):
    m = mask.astype(bool)                       # mask comes back as float
    pred = fp[m].argmax(axis=-1)
    true = ft[m].argmax(axis=-1)
    seqs[name] = "".join(alphabet[i] for i in pred)
    print(f"{name}  len={len(pred)}  recovery={(pred == true).mean():.3f}")
    print("pred:", seqs[name])
    print("real:", "".join(alphabet[i] for i in true))


7yxu  len=229  recovery=0.913
pred: EVQLLESGGGLIQPGGSLRLSCAASGFTFSSGKMHWVRQAPGKGLEWISSISSGSGVTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKDMYLGSFYLIFYWGQGTLVTVSSDIQMTQSPSSLSASVGDRVTITCRASQSISIYLAWYQQKPGKAPKLLIYAASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQYSSYFPTFGLGTKLEIKR
real: EVQLLESGGGLVQPGGSLRLSCAASGFTFSYGSMYWVRQAPGKGLEWVSSISSGSGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARSSYYGSYYSIDYWGQGTLVTVSSDIQMTQSPSSLSASVGDRVTITCRASQSISSYLNWYQQKPGKAPKLLIYAASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQYYDNLPTFGQGTKLEIKR


### Running inf using input  data as a pkl


If we want to to re-run inference multiple times using the same data, we can save the data as a pickle file so that it can be quickly loaded in during inference. 

once saved set load_pkl to fpath of pkl dataset to load.

In [12]:
# load model for inf need to also load weights.
for batch in dataloader:
    out = model.sample(batch.clone().to(device), closure=True)
    break


In [13]:
with open('data/raw_data/example_ds.pkl', 'wb') as f:
    pkl.dump(custom_ds, f)

In [14]:
args.data.load_pkl = 'data/raw_data/example_ds.pkl'

In [15]:
if args.data.load_pkl:
    with open(args.data.load_pkl, 'rb') as f:
        dataset = pkl.load(f) 